## Register High-res H&E, Cy-IF & DESI WSI series W/ VALIS

In [ ]:
import os
import sys
import gc

import gcsfs
import tifffile
import numpy as np
import pandas as pd
import seaborn as sns


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

In [ ]:
sys.path.append('..')
import registration, IO

### H&E alignment

#### Data loading & alignment

In [ ]:
data_path = '../data/he_imgs/'
res_path = '../data/he_imgs/res/'
ref_slide = '09.tif'

In [ ]:
registrar, _, _ = registration.run_valis(src_dir=data_path,
                                         res_dir=res_path,
                                         ref_slide=ref_slide,
                                         micro=False,
                                         kill_jvm=True)

In [ ]:
# Generate overlayed H&E WSI series
aligned_imgs = [tifffile.imread(os.path.join(res_path, 'registered_slides', f))
                for f in sorted(os.listdir(os.path.join(res_path, 'registered_slides')))
                if f[-8:] == 'ome.tiff']

aligned_imgs = np.array(aligned_imgs)
aligned_imgs = aligned_imgs.transpose((3,0,1,2))
tifffile.imwrite(os.path.join(res_path, 'valis_stacked.ome.tif'), aligned_imgs, metadata={'axes': 'CZYX'})

del aligned_imgs

#### Evaluation

- Comparison of registration results btw `rigid`, `non-rigid` & `micro-registration`

In [ ]:
# Per-slice errors before & after alignment
summary_df = pd.read_csv(os.path.join(res_path, 'data/_summary.csv'), index_col=[0])
summary_df.head()

In [ ]:
# rTRE score: relative Target Registration Error
# (avg. normed. dist btw landmarks from "src" to "dst")

nslides = summary_df.shape[0]

rTRE_df = pd.DataFrame({
    'name': nslides*['orig'] + nslides*['rigid'] + nslides*['non-rigid'],
    'rTRE': summary_df['original_rTRE'].to_list() + summary_df['rigid_rTRE'].to_list() + summary_df['non_rigid_rTRE'].to_list()
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='name', y='rTRE', data=rTRE_df,
               palette=['cyan','g','orange'], ax=ax)
ax.set_title('relative Target Registration Error')
plt.show()

In [ ]:
nr_margin_df = pd.DataFrame({
    'name': nslides*['rigid'] + nslides*['non-rigid'],
    'rTRE': summary_df['rigid_rTRE'].to_list() + summary_df['non_rigid_rTRE'].to_list()
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='name', y='rTRE', data=nr_margin_df,
            palette=['g','orange'], ax=ax)

ax.set_title('Rigid vs. Non-rigid')
plt.show()

Both non-rigid & micro-registration introduces artifacts outside the registered bounding-box (likely introduced from VALIS's pyramid schemes); Trim off before using the images)

---

### Post-CyCIF H&E alignment

Load images from gcloud

In [ ]:
def disp_chans(img, title=None, ncols=4):
    depth = len(img)
    nrows = depth // ncols if depth % ncols == 0 else depth // ncols + 1
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(3*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= depth:
                axes[r, c].axis('off')
                continue
            axes[r, c].imshow(img[idx])
            idx += 1
            
    fig.tight_layout()
    fig.suptitle(title, y=1.01)
    plt.show()

In [ ]:
CREDENTIAL_PATH = "../data/azizilab-cell-segmentation-05f1a1125db2.json"
PROJECT_ID = 'azizilab-cell-segmentation'
BUCKET_ID = 'liver_3d'
HOME_PATH = 'CyIF/H&E_post_CyIF'

he_reader = IO.GcloudReader(CREDENTIAL_PATH,
                            PROJECT_ID,
                            BUCKET_ID,
                            HOME_PATH)


he_imgs = he_reader.load_imgs()

In [ ]:
del 

In [ ]:
for img in he_imgs:
    plt.figure()
    plt.imshow(img)
    plt.show()


In [ ]:
disp_chans(he_imgs)

In [ ]:
# DEBUG: whether VALIS support google-cloud reading

### CyCIF alignment

#### Data loading & alignment

- **DEBUG**: few tissues have severe AF issues; Current aiming to register **slide 1-3**

In [ ]:
from skimage import io

In [ ]:
%ls ../data/cycif/Cy-IF_downsample_adj/

In [ ]:
# Extract DAPI channels

# data_path = '../data/cycif/Cy-IF_downsample_adj/'
# res_path = '../data/cycif/Cy-IF_downsample_aln/'
# dapi_path = '../data/cycif/Cy-IF_downsample_dapi/'
# if not os.path.exists(dapi_path):
#     os.makedirs(dapi_path, exist_ok=True)

# filenames = [f for f in sorted(os.listdir(data_path))
#              if f[-3:] == 'tif']

# for f in filenames:
#     filename = f.split('.')[0]+'.tif'
#     dapi = tifffile.imread(os.path.join(data_path, f))[0]
#     tifffile.imwrite(os.path.join(dapi_path, filename), dapi)

# del f, filename, dapi

In [ ]:
# Alignment w/ a reference X work well

data_path = '../data/cycif/Cy-IF_downsample_adj/'
# dapi_path = '../data/cycif/Cy-IF_downsample_dapi/'
res_path = '../data/cycif/Cy-IF_downsample_aln/'
ref_slide = 'CyIF_tiss_10.ome.tif'
# ref_slide = None

series = 2
do_micro = False
aln_to_ref = False 
# micro_res = 1000

registrar, _, _ = registration.run_valis(src_dir=data_path,
                                         res_dir=res_path,
                                         ref_slide=ref_slide,
                                         micro=do_micro,
                                         image_type='fluorescence',
                                         kill_jvm=True)

#### Evaluation

In [ ]:
# Per-slice errors before & after alignment
summary_df = pd.read_csv(os.path.join(res_path, 'data/_summary.csv'), index_col=[0])
summary_df.head()

In [ ]:
nslides = summary_df.shape[0]

rTRE_df = pd.DataFrame({
    'name': nslides*['orig'] + nslides*['rigid'] + nslides*['non-rigid'],
    'rTRE': summary_df['original_rTRE'].to_list() + summary_df['rigid_rTRE'].to_list() + summary_df['non_rigid_rTRE'].to_list()
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(x='name', y='rTRE', data=rTRE_df,
               palette=['cyan','g','orange'], ax=ax)
ax.set_title('relative Target Registration Error')
plt.show()

---